# Hierarchical (Multilevel) Models

## Overview

Hierarchical models (also called multilevel models) account for data that is grouped or nested — sites within catchments, repeated measures within individuals, species within families. They partially pool information across groups: groups with little data borrow strength from the overall distribution.

**The partial pooling continuum:**

| Approach | Description | Problem |
|---|---|---|
| No pooling | Fit each group separately | Overfits small groups |
| Complete pooling | Ignore groups entirely | Ignores group differences |
| Partial pooling | Hierarchical prior across groups | Best of both |

**Partial pooling:** each group's estimate is shrunk toward the global mean, with the degree of shrinkage proportional to the group's sample size and variance. Small groups shrink more.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf

rng = np.random.default_rng(42)
# 8 catchments, variable sample sizes, shared ecology
n_catchments = 8
true_grand_mean = 20.0
true_catchment_sd = 4.0
true_resid_sd = 3.0
true_means = rng.normal(true_grand_mean, true_catchment_sd, n_catchments)
ns = rng.integers(5, 40, n_catchments)  # variable n per catchment
data_list = []
for i, (mu_i, n_i) in enumerate(zip(true_means, ns)):
    richness = rng.normal(mu_i, true_resid_sd, n_i)
    data_list.append(pd.DataFrame({"catchment": f"C{i+1}", "richness": richness,
                                    "true_mean": mu_i}))
df = pd.concat(data_list, ignore_index=True)
print(df.groupby("catchment").agg(n=("richness","count"),
      mean=("richness","mean"), sd=("richness","std")).round(2))

---
## No Pooling vs Complete Pooling

In [ ]:
# No pooling: separate mean for each catchment
no_pool = df.groupby("catchment")["richness"].agg(["mean","sem","count"])
no_pool.columns = ["mean_np","se_np","n"]
no_pool["ci_lo_np"] = no_pool["mean_np"] - 1.96*no_pool["se_np"]
no_pool["ci_hi_np"] = no_pool["mean_np"] + 1.96*no_pool["se_np"]
# Complete pooling: single grand mean
complete_mean = df["richness"].mean()
complete_se   = df["richness"].sem()
print(f"Complete pooling: mean={complete_mean:.2f}, SE={complete_se:.2f}")
print("\nNo pooling estimates:")
print(no_pool[["n","mean_np","se_np"]].round(3))

---
## Partial Pooling via Mixed-Effects Model

In [ ]:
# Linear mixed model: catchment as random intercept
mixed = smf.mixedlm("richness ~ 1", df, groups=df["catchment"]).fit(reml=True)
print(mixed.summary())
# Extract random effects (catchment-specific deviations)
rand_effects = mixed.random_effects
partial_pool_means = {
    cat: mixed.fe_params["Intercept"] + rand_effects[cat]["Group"]
    for cat in rand_effects
}
print("\nPartial pooling means:")
for cat, est in sorted(partial_pool_means.items()):
    n_cat = no_pool.loc[cat,"n"]
    true_m = df[df["catchment"]==cat]["true_mean"].iloc[0]
    print(f"  {cat} (n={n_cat:2d}): partial={est:.2f}, no-pool={no_pool.loc[cat,'mean_np']:.2f}, true={true_m:.2f}")

---
## Visualising Shrinkage

In [ ]:
catchments = sorted(df["catchment"].unique())
fig, ax = plt.subplots(figsize=(10,6))
true_m = [df[df["catchment"]==c]["true_mean"].iloc[0] for c in catchments]
np_means = [no_pool.loc[c,"mean_np"] for c in catchments]
pp_means = [partial_pool_means[c] for c in catchments]
ns_c = [no_pool.loc[c,"n"] for c in catchments]
x = np.arange(len(catchments))
ax.scatter(x, true_m,  s=100, color="navy",      zorder=5, label="True mean", marker="D")
ax.scatter(x, np_means, s=80, color="#e74c3c",   zorder=4, label="No pooling", marker="o")
ax.scatter(x, pp_means, s=80, color="#4fffb0",   zorder=4, label="Partial pooling", marker="s")
ax.axhline(complete_mean, color="grey", lw=1.5, linestyle="--", label="Complete pooling")
for i, (np_m, pp_m, n_i) in enumerate(zip(np_means, pp_means, ns_c)):
    ax.annotate(f"n={n_i}", (i, np_m+0.3), ha="center", fontsize=8, color="#e74c3c")
ax.set_xticks(x); ax.set_xticklabels(catchments)
ax.set_ylabel("Species richness estimate")
ax.set_title("Partial Pooling: Small Groups Shrink Toward Grand Mean")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Demonstrate shrinkage mathematically
print("Shrinkage analysis:")
print(f"Grand mean (complete pooling): {complete_mean:.2f}")
grand_var  = mixed.cov_re.values[0,0]
resid_var  = mixed.scale
for cat in catchments:
    n_i = no_pool.loc[cat,"n"]
    # Shrinkage factor lambda: proportion pulled toward grand mean
    lam = resid_var / (resid_var + n_i * grand_var)
    print(f"  {cat} n={n_i:2d}: shrinkage={lam:.3f} -> {lam:.0%} toward grand mean")
print("\nSmall n -> high shrinkage; large n -> estimate stays near observed mean")

---

## Common Pitfalls

**1. Ignoring group structure and using complete pooling**  
Grouped data often has within-group correlation. Ignoring groups (complete pooling) treats all observations as independent, underestimates standard errors, and produces overconfident inference. Always test whether adding random intercepts substantially improves fit.

**2. Using no-pooling with small groups**  
Fitting a separate model for each group is unbiased but high-variance for small groups. A catchment with n=5 has a very uncertain mean estimate that can be substantially improved by partial pooling from other catchments.

**3. Not checking the random effects distribution assumption**  
Mixed models assume random effects are normally distributed. For categorical groupings with a small number of levels (< 5), this assumption cannot be checked. With 5–20 levels, plot the estimated random effects and check for obvious non-normality.

**4. Reporting random effect variances without contextualising them**  
The random effects variance (between-group SD) is scientifically important — it quantifies how much the catchments differ. Always report and interpret the between-group SD alongside the fixed effects, and compute the intraclass correlation coefficient (ICC = between / (between + within)).

**5. Confusing fixed effects and random effects specification**  
Fixed effects are predictors whose specific levels you want to estimate (treatment, elevation). Random effects are grouping factors where you want to model the distribution of levels (catchment, site, year). A predictor should be random when the levels in the data are a sample from a broader population of possible levels.
---
*python_methods_library - Samantha McGarrigle*